<a href="https://colab.research.google.com/github/ShahdOmar1123/CodeAlphaTasks/blob/main/chromaDBandFaiss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets chromadb==0.4.22 sentence-transformers==2.3.1 faiss-cpu

In [2]:
from datasets import load_dataset
qna_data=load_dataset("sadeem-ai/arabic-qna")
news_data=load_dataset("arbml/SANAD")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


In [3]:
print(news_data.column_names)

{'train': ['Article', 'label']}


In [4]:
news_data= news_data.filter(lambda example: len(example["Article"]) >= 100)
news_data

DatasetDict({
    train: Dataset({
        features: ['Article', 'label'],
        num_rows: 141603
    })
})

In [5]:
news_data=news_data.shuffle(seed=42)

In [6]:
qna_data = qna_data.filter(lambda example: example["has_answer"] == True)

In [7]:
doc_text = list(qna_data["train"]["text"]) + list(news_data["train"]["Article"][:30000])
doc_question = qna_data["train"]["question"]

In [8]:
len(doc_text)

34037

In [9]:
meta_data=[{
    "source":rec['source'],
    "title":rec['title']
    }
  for rec in qna_data["train"]
    ]
meta_data += [
    {
        "source": "",
        "title": "",
    }
    for i in range(30_000)
 ]

In [10]:
len(meta_data)

34037

In [11]:
doc_id=[
    str(i)
    for i in range(len(doc_text))
]

In [12]:
len(doc_id)

34037

In [13]:
doc_id[50],doc_question[50],meta_data[50]

('50',
 'من أين استخلص المعدن لصنع الميداليات الأولمبية؟',
 {'source': 'https://ar.wikipedia.org/wiki?curid=8379033',
  'title': 'جدول ميداليات الألعاب الأولمبية الصيفية 2020'})

In [14]:
import torch
print(torch.cuda.is_available())

True


In [15]:
from sentence_transformers import SentenceTransformer

model_id = "sentence-transformers/distiluse-base-multilingual-cased-v2"
dim = 512
device = "cuda:0"
model=SentenceTransformer(model_id,device=device)

In [16]:
encoded_doc=model.encode(doc_text,show_progress_bar=True)

Batches:   0%|          | 0/1064 [00:00<?, ?it/s]

In [17]:
doc_question = list(doc_question)
encoded_qus=model.encode(doc_question,show_progress_bar=True)

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

In [18]:
#!pip uninstall -y numpy
#!pip install numpy==1.26.4

In [19]:
import chromadb
chroma_client = chromadb.PersistentClient(path="./chromadb-ar-docs")
collection = chroma_client.create_collection(
    name="ar_docs_34k",
    metadata={"hnsw:space": "cosine"}
)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [20]:
collection.add(
    documents=doc_text,
    metadatas=meta_data,
    embeddings=encoded_doc,
    ids=doc_id
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [21]:
question="ما السبب في صغر الأسنان بالمقارنة مع حجم الفكين؟"
question_embedding=model.encode(question)
results=collection.query( query_embeddings=question_embedding.tolist(),
    n_results=3)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [22]:
results

{'ids': [['397', '1534', '29116']],
 'distances': [[0.419485867023468, 0.419485867023468, 0.5358186960220337]],
 'metadatas': [[{'source': 'https://ar.wikipedia.org/wiki?curid=7483665',
    'title': 'صغر الأسنان'},
   {'source': 'https://ar.wikipedia.org/wiki?curid=7483665',
    'title': 'صغر الأسنان'},
   {'source': '', 'title': ''}]],
 'embeddings': None,
 'documents': [['جميع الأسنان ذات حجم طبيعي ولكنها تبدو صغيرة بسبب ضخامة الفكين. قد يكون الصغر النسبي المعمم نتيجة وراثة فك كبير من أحد الوالدين، وأسنان ذات حجم طبيعي من الآخر.',
   'جميع الأسنان ذات حجم طبيعي ولكنها تبدو صغيرة بسبب ضخامة الفكين. قد يكون الصغر النسبي المعمم نتيجة وراثة فك كبير من أحد الوالدين، وأسنان ذات حجم طبيعي من الآخر.',
   'قليلون جداً هم مَن يتمتعون بطاقم أسنان مثالي دون أية تدخلات. وعندما تسقط الأسنان اللبنية تنمو الأسنان الدائمة في الغالب بشكل غير منتظم، ولكن متى يتطلب الأمر العلاج؟    وللإجابة على هذا السؤال، قال طبيب الأسنان الألماني ديرك كروب إن اعوجاج الأسنان قد لا ينتج عنه أية مشاكل صحية، ولكنه في الوق

In [23]:
import faiss
import numpy as np
from copy import deepcopy

In [24]:
norm_encoded_docs = deepcopy(encoded_doc)
faiss.normalize_L2(norm_encoded_docs)

In [25]:
doc_id = np.array(doc_id).astype('int64')
faiss_index = faiss.IndexIDMap( faiss.IndexFlatIP(dim) )

faiss_index.add_with_ids( norm_encoded_docs, doc_id)

In [26]:
question = "ما السبب في صغر الأسنان بالمقارنة مع حجم الفكين؟"
question_embed = model.encode([question])

faiss.normalize_L2(question_embed)

results = faiss_index.search(question_embed, 3)
print(results)

(array([[0.5805141 , 0.5805141 , 0.46418092]], dtype=float32), array([[ 1534,   397, 29116]]))
